# RiceChem systems profiling analysis

This notebook reproduces the public aggregate systems analysis without running a model or loading RiceChem rows. It separates **measured deployment results** from the controlled concurrency-1 CPU/GPU/TPU profile. The checkpoint path and test manifest matched across that profile; missing checkpoint/image hashes and unavailable TPU device telemetry remain explicit limitations.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'
assert (RESULTS / 'results_report.json').exists(), 'Run this notebook from the repository root or notebooks/.'

## Observed serving results

Every row below used the frozen 861-decision canonical test. Throughput is an observed property of the complete serving path—not a model-size or hardware ceiling.

In [2]:
report = json.loads((RESULTS / 'results_report.json').read_text())
serving = pd.DataFrame([{
    'label': row['label'],
    'n': row['n'],
    'agreement_pct': 100 * row['acc_minus1'],
    'wall_seconds': row['wall_secs'],
    'decisions_per_second': row['throughput_per_sec'],
} for row in report['cells']])
assert set(serving['n']) == {861}
focus = serving[serving['label'].isin(['ft-tpu-bare', 'ft-gpu-bare', 'ft27b-bare', 'base27b-bare', 'opus5-cli-bare'])].copy()
focus.sort_values('decisions_per_second', ascending=False)

,label,n,agreement_pct,wall_seconds,decisions_per_second
2,ft-tpu-bare,861,72.13,11.7,73.340
3,ft-gpu-bare,861,71.78,17.0,50.680
4,ft27b-bare,861,83.28,645.2,1.330
0,opus5-cli-bare,861,80.49,675.1,1.275
5,base27b-bare,861,70.96,693.8,1.240


In [3]:
label_map = {
    'ft-tpu-bare': 'Gemma 4B FT · TPU v5e-8 · vLLM',
    'ft-gpu-bare': 'Gemma 4B FT · A100 · vLLM',
    'ft27b-bare': 'Gemma 27B FT · A100 · unbatched',
    'base27b-bare': 'Gemma 27B base · A100 · unbatched',
    'opus5-cli-bare': 'Claude Opus 5 · API via CLI',
}
plot_df = focus.assign(display=focus['label'].map(label_map)).sort_values('decisions_per_second')
ax = plot_df.plot.barh(x='display', y='decisions_per_second', legend=False, color='#4285F4', figsize=(9, 4.5))
ax.set(title='Observed grading throughput', xlabel='Rubric decisions per second', ylabel='')
ax.set_xscale('log')
plt.tight_layout()

## Training telemetry

The timing receipts distinguish optimizer-loop time from total elapsed time. This avoids calling model load, data preparation, or post-train work part of the training loop.

In [4]:
timing_rows = []
for path in sorted(RESULTS.glob('training_timing_*.json')):
    row = json.loads(path.read_text())
    timing_rows.append({
        'lane': row['lane'],
        'devices': row['devices'],
        'train_minutes': row['train_secs'] / 60,
        'total_minutes': row['total_wall_secs'] / 60,
        'real_tokens_per_second': row['real_tokens_per_sec'],
        'model_load_seconds': row['model_load_secs'],
    })
training = pd.DataFrame(timing_rows).sort_values('real_tokens_per_second', ascending=False)
training

,lane,devices,train_minutes,total_minutes,real_tokens_per_second,model_load_seconds
2,tpu-v5e,8,43.716667,44.353333,1689.6,12.4
1,gpu-a100,1,90.706667,93.213333,815.8,145.9
0,gpu-a100-27b-qlora,1,184.325000,190.525000,267.6,359.5


In [5]:
gpu4 = training.loc[training['lane'].eq('gpu-a100')].iloc[0]
tpu4 = training.loc[training['lane'].eq('tpu-v5e')].iloc[0]
summary = pd.Series({
    'TPU/A100 4B token-throughput ratio': tpu4.real_tokens_per_second / gpu4.real_tokens_per_second,
    'A100/TPU 4B training-loop time ratio': gpu4.train_minutes / tpu4.train_minutes,
    'A100/TPU 4B total-elapsed ratio': gpu4.total_minutes / tpu4.total_minutes,
    '27B canonical-test turnaround (minutes)': 861 / float(serving.loc[serving.label.eq('ft27b-bare'), 'decisions_per_second'].iloc[0]) / 60,
})
summary.round(2)

TPU/A100 4B token-throughput ratio          2.07
A100/TPU 4B training-loop time ratio        2.07
A100/TPU 4B total-elapsed ratio             2.10
27B canonical-test turnaround (minutes)    10.79
dtype: float64

## Controlled CPU/GPU/TPU matrix

The public receipt contains completed full-test rows only. The notebook validates all 861 decisions, displays measured fields, and leaves unavailable telemetry missing rather than estimating it.

In [6]:
hardware_path = RESULTS / 'hardware_comparison.csv'
required = ['hardware', 'checkpoint', 'n', 'concurrency', 'decisions_per_second', 'wall_seconds', 'latency_p50_ms', 'latency_p95_ms', 'agreement_pct', 'mean_utilization_pct', 'peak_memory_gb', 'cost_usd', 'telemetry_source', 'status']
if hardware_path.exists():
    hardware = pd.read_csv(hardware_path)
    missing = [column for column in required if column not in hardware.columns]
    assert not missing, f'Missing columns: {missing}'
    assert set(hardware['n']) == {861}, 'Controlled results must use all 861 decisions.'
    completed_hardware = hardware[hardware['status'].eq('complete')].copy()
    display(completed_hardware.sort_values(['concurrency', 'decisions_per_second'], ascending=[True, False]))
else:
    print('Controlled same-checkpoint hardware matrix unavailable; no values inferred.')

,hardware,checkpoint,n,concurrency,decisions_per_second,wall_seconds,latency_p50_ms,latency_p95_ms,agreement_pct,mean_utilization_pct,peak_memory_gb,cost_usd,telemetry_source,status
3,TPU v5e-8,merged-ft-gemma3-4b (hash unavailable),861,1,11.39,75.6,82.2,104.0,72.36,NaN,NaN,NaN,host CPU and memory only; TPU utilization and ...,complete
1,A100 40 GB,merged-ft-gemma3-4b (hash unavailable),861,1,8.54,100.8,116.1,128.6,72.01,19.5,35.6,NaN,coarse NVML GPU utilization and VRAM series du...,complete
0,16-vCPU,merged-ft-gemma3-4b (hash unavailable),861,1,0.28,3054.2,3315.3,5571.6,72.71,95.1,18.0,NaN,process CPU and RSS during the full run,complete
2,A100 40 GB,merged-ft-gemma3-4b (hash unavailable),861,24,123.87,7.0,187.7,225.8,72.13,NaN,NaN,NaN,run shorter than telemetry sampling interval,complete
4,TPU v5e-8,merged-ft-gemma3-4b (hash unavailable),861,24,98.41,8.7,219.6,516.4,72.24,NaN,NaN,NaN,TPU utilization and HBM unavailable,complete


In [7]:
c1 = completed_hardware.loc[completed_hardware['concurrency'].eq(1)].copy()
assert set(c1['hardware']) == {'16-vCPU', 'A100 40 GB', 'TPU v5e-8'}
cpu_rate = float(c1.loc[c1.hardware.eq('16-vCPU'), 'decisions_per_second'].iloc[0])
c1['speedup_vs_cpu'] = c1['decisions_per_second'] / cpu_rate
display(c1[['hardware', 'decisions_per_second', 'wall_seconds', 'latency_p50_ms', 'latency_p95_ms', 'mean_utilization_pct', 'peak_memory_gb', 'speedup_vs_cpu']].round(2))
ax = c1.sort_values('decisions_per_second').plot.barh(x='hardware', y='decisions_per_second', legend=False, color=['#7A8797', '#7A55C4', '#4285F4'], figsize=(8, 3.8))
ax.set(title='Controlled full-test throughput · concurrency 1', xlabel='Rubric decisions per second', ylabel='')
plt.tight_layout()

,hardware,decisions_per_second,wall_seconds,latency_p50_ms,latency_p95_ms,mean_utilization_pct,peak_memory_gb,speedup_vs_cpu
0,16-vCPU,0.28,3054.2,3315.3,5571.6,95.1,18.0,1.00
1,A100 40 GB,8.54,100.8,116.1,128.6,19.5,35.6,30.50
3,TPU v5e-8,11.39,75.6,82.2,104.0,NaN,NaN,40.68


## Interpretation contract

1. Compare hardware as hardware only when checkpoint, prompt, parser, output limit, batching, and concurrency match.
2. Report unavailable utilization or memory telemetry as unavailable.
3. Do not interpret the unbatched 27B endpoint as an A100 hardware ceiling.
4. Treat 4B throughput as operationally irrelevant if its grading agreement creates more human review work than it removes.